# Superquadric decomposition

Plug a point cloud into `PC_PATH` below (in `1. Configuration`) and run the cells top to bottom.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import pipeline
from src.superquadrics import superquadric_mesh as supmesh
from src.visualizations import plot as vis

print("repo root :", ROOT)
print("pipeline  :", pipeline.__file__)
print("MASTER_SEED:", pipeline.MASTER_SEED)

import pyvista as pv

try:
    pv.set_jupyter_backend("trame")  # interactive 3D, inline in the notebook
except ImportError:
    pv.set_jupyter_backend("none")  # separate blocking desktop window

print("pyvista    :", pv.__version__, "| jupyter_backend:", pv.global_theme.jupyter_backend)

repo root : C:\Users\Alessandro\Desktop\Research\GEOPRIDE\superquadric-decomposition-public
pipeline  : C:\Users\Alessandro\Desktop\Research\GEOPRIDE\superquadric-decomposition-public\src\pipeline.py
MASTER_SEED: 12345
pyvista    : 0.49.0 | jupyter_backend: trame


## 1. Configuration

`PC_PATH` is the only thing you normally need to change. Any format `trimesh` reads works
(`.ply`, `.stl`, `.glb`, `.obj`, ...). If a sibling file named `normals_<filename>` exists it
is used for normals; otherwise they are estimated from the mesh or knn-estimated for a bare
point cloud.

In [2]:
# -- plug your point cloud in here -------------------------------------------
PC_PATH = ROOT / "data" / "point_clouds" / "bird.stl"

# -- sweep size --------------------------------------------------------------
ALGORITHMS   = ["vanilla", "lo-ransac", "gc-ransac", "gair-ransac"]
OUTLIER_FRAC = 0.01    # fraction of uniform outliers to inject
SEED         = 12345   # this notebook's own seed;
N_TRIALS = 3

# -- set-cover ---------------------------------------------------------------
K            = 5       # max superquadrics the set-cover step may select - set it per cloud

# -- knobs mirrored from src/pipeline.py; overridden here
pipeline.MAX_LOAD_PTS = 8000
pipeline.VISUALIZE    = False   # visualisation is done explicitly in section 4
pipeline.K            = K       # kept in sync so the module default matches this notebook
pipeline.MAX_MODELS   = 10      # max models one RANSAC run may find (candidate pool size)

assert PC_PATH.exists(), f"point cloud not found: {PC_PATH}"
print(f"cloud      : {PC_PATH.name}")
print(f"algorithms : {ALGORITHMS}")
print(f"trials     : {N_TRIALS}   outlier_frac: {OUTLIER_FRAC}   seed: {SEED}")
print(f"max k      : {K}   max_models: {pipeline.MAX_MODELS}")

cloud      : bird.stl
algorithms : ['vanilla', 'lo-ransac', 'gc-ransac', 'gair-ransac']
trials     : 3   outlier_frac: 0.01   seed: 12345
max k      : 5   max_models: 10


## 2. Load

`pipeline.load_point_cloud` normalises the point cloud and injects outliers.

In [3]:
points_clean, normals_clean = pipeline.load_point_cloud(PC_PATH)
n_clean = len(points_clean)
pc_cfg  = pipeline.load_pc_config(PC_PATH.stem)

print(f"\n{n_clean} points loaded")
print(f"config     : {pc_cfg}")
if not pc_cfg:
    print("  (no entry in data/pc_configs.json -> src/pipeline.py module defaults apply)")
print(f"threshold  : {pc_cfg.get('threshold', pipeline.THRESHOLD)}")

  bounding box after normalization: min=[0. 0. 0.]  max=[0.50248776 1.         0.98300561]

1439 points loaded
config     : {'threshold': 0.03, 'mss_sample_size': 50, 'mss_pts_per_patch': 5, 'mss_voxel_spacing_factor': 150}
threshold  : 0.03


In [4]:
rng         = np.random.default_rng(SEED)
trial_seeds = [int(rng.integers(0, 2**31)) for _ in range(N_TRIALS)]

points, normals = pipeline.corrupt_point_cloud(points_clean.copy(), normals_clean.copy(), rng)

n_outliers = int(n_clean * OUTLIER_FRAC)
if n_outliers > 0:
    lo, hi      = points_clean.min(axis=0), points_clean.max(axis=0)
    outlier_pts = rng.uniform(lo, hi, (n_outliers, 3))
    outlier_nrm = rng.normal(0, 1, (n_outliers, 3))
    outlier_nrm /= np.linalg.norm(outlier_nrm, axis=1, keepdims=True)
    points  = np.vstack([points,  outlier_pts])
    normals = np.vstack([normals, outlier_nrm])

print(f"{n_clean} clean + {n_outliers} outliers = {len(points)} points")
print(f"trial seeds: {trial_seeds}")

1439 clean + 14 outliers = 1453 points
trial seeds: [1501552846, 488200390, 1693606511]


## 3. Run the algorithms

In [5]:
import time

rows, models_by = [], {}

for trial, seed in enumerate(trial_seeds):
    for algo in ALGORITHMS:
        t0  = time.perf_counter()
        res = pipeline.run_one(points, normals, points_clean, n_clean, algo, seed, pc_cfg)
        if res is None:
            print(f"  trial {trial} {algo:12s} -> no model found, skipped")
            continue
        models_by[(trial, algo)] = res["models"]
        rows.append({
            "algo": algo, "trial": trial, "seed": seed,
            **{k: v for k, v in res.items() if k != "models"},
        })
        print(f"  trial {trial} {algo:12s} CD={res['chamfer']:.4f}"
              f"  COV={res['cd_coverage']:.4f}  ACC={res['cd_accuracy']:.4f}"
              f"  HD={res['hausdorff']:.4f}  CLF={res['classification_rate']:.3f}"
              f"  models={res['n_models']}  {time.perf_counter() - t0:.1f}s")

df = pd.DataFrame(rows)
print(f"\n{len(df)} runs completed")
df

    correctly_classified=1216/1453  (83.7%)
  trial 0 vanilla      CD=0.1368  COV=0.0151  ACC=0.1218  HD=0.7253  CLF=0.837  models=5  16.3s
    correctly_classified=1264/1453  (87.0%)
  trial 0 lo-ransac    CD=0.0972  COV=0.0149  ACC=0.0824  HD=0.5387  CLF=0.870  models=4  23.8s
    correctly_classified=721/1453  (49.6%)
  trial 0 gc-ransac    CD=0.0956  COV=0.0710  ACC=0.0246  HD=0.2901  CLF=0.496  models=1  44.8s
    correctly_classified=1169/1453  (80.5%)
  trial 0 gair-ransac  CD=0.0388  COV=0.0138  ACC=0.0250  HD=0.2008  CLF=0.805  models=5  24.4s
    correctly_classified=1237/1453  (85.1%)
  trial 1 vanilla      CD=0.1279  COV=0.0152  ACC=0.1127  HD=0.5384  CLF=0.851  models=5  7.8s
    correctly_classified=1274/1453  (87.7%)
  trial 1 lo-ransac    CD=0.1323  COV=0.0148  ACC=0.1175  HD=0.5968  CLF=0.877  models=5  17.7s
    correctly_classified=853/1453  (58.7%)
  trial 1 gc-ransac    CD=0.1045  COV=0.0260  ACC=0.0785  HD=0.5556  CLF=0.587  models=2  22.8s
    correctly_classifie

,algo,trial,seed,chamfer,cd_coverage,cd_accuracy,hausdorff,classification_rate,runtime_s,n_models,n_inliers,n_local_opts
0,vanilla,0,1501552846,0.136834,0.015059,0.121775,0.725316,0.836889,15.690106,5,1210,0
1,lo-ransac,0,1501552846,0.097248,0.014894,0.082354,0.538725,0.869924,23.623751,4,1257,8
2,gc-ransac,0,1501552846,0.095645,0.071040,0.024605,0.290050,0.496215,44.721690,1,707,39
3,gair-ransac,0,1501552846,0.038790,0.013825,0.024965,0.200755,0.804542,24.166026,5,1156,13
4,vanilla,1,488200390,0.127919,0.015196,0.112723,0.538446,0.851342,7.535642,5,1233,0
5,lo-ransac,1,488200390,0.132261,0.014760,0.117500,0.596771,0.876807,17.405796,5,1272,8
6,gc-ransac,1,488200390,0.104466,0.025999,0.078467,0.555644,0.587061,22.738749,2,845,21
7,gair-ransac,1,488200390,0.039148,0.020284,0.018864,0.139167,0.710943,38.682747,4,1019,23
8,vanilla,2,1693606511,0.131401,0.015971,0.115430,0.771780,0.848589,10.739626,4,1224,0
9,lo-ransac,2,1693606511,0.127233,0.013772,0.113460,0.895805,0.913283,15.820310,4,1319,8


In [6]:
# mean / std per algorithm, in ALGORITHMS order
METRICS = ["chamfer", "cd_coverage", "cd_accuracy", "hausdorff",
           "classification_rate", "runtime_s", "n_models"]

summary = (df.groupby("algo")[METRICS]
             .agg(["mean", "std"])
             .reindex([a for a in ALGORITHMS if a in set(df["algo"])])
             .round(4))
summary

chamfer         cd_coverage         cd_accuracy         hausdorff  \
               mean     std        mean     std        mean     std      mean   
algo                                                                            
vanilla      0.1321  0.0045      0.0154  0.0005      0.1166  0.0046    0.6785   
lo-ransac    0.1189  0.0189      0.0145  0.0006      0.1044  0.0192    0.6771   
gc-ransac    0.0927  0.0134      0.0378  0.0292      0.0550  0.0276    0.4308   
gair-ransac  0.0405  0.0027      0.0168  0.0033      0.0237  0.0044    0.1795   

                    classification_rate         runtime_s          n_models  \
                std                mean     std      mean      std     mean   
algo                                                                          
vanilla      0.1235              0.8456  0.0077   11.3218   4.1083   4.6667   
lo-ransac    0.1916              0.8867  0.0233   18.9500   4.1245   4.3333   
gc-ransac    0.1335              0.6091  0.1253   31.4069  11.7049   2.0000   
gair-ransac  0.0350              0.7598  0.0469   35.9148  10.6383   4.6667   

                     
                std  
algo                 
vanilla      0.5774  
lo-ransac    0.5774  
gc-ransac    1.0000  
gair-ransac  0.5774

## 4. Visualise

Renders **inline** when `trame-pyvista` is installed (the setup cell reports which backend is
active), otherwise opens a blocking PyVista desktop window.

In [7]:
VIEW_TRIAL = 0
VIEW_ALGO  = "gair-ransac"   # any entry of ALGORITHMS

PALETTE = ["lightgreen", "orange", "violet", "cyan", "yellow",
           "red", "lime", "pink", "gold", "turquoise"]

models = models_by.get((VIEW_TRIAL, VIEW_ALGO))
if not models:
    print(f"no models stored for trial {VIEW_TRIAL} / {VIEW_ALGO}")
else:
    meshes = [supmesh.superquadric_mesh(m) for m in models]
    colors = [PALETTE[i % len(PALETTE)] for i in range(len(meshes))]
    print(f"{VIEW_ALGO} trial {VIEW_TRIAL}: {len(models)} superquadric(s)"
          " - close the window to continue")
    vis.show_mesh_and_points(
        meshes, pts=points, point_size=5, colors=colors,
        models=models, treshold=pc_cfg.get("threshold", pipeline.THRESHOLD),
    )

gair-ransac trial 0: 5 superquadric(s) - close the window to continue


Widget(value='<iframe src="http://localhost:64958/index.html?ui=P_0x1ecc0088f80_0&reconnect=auto" class="pyvis…

Widget(value='<iframe src="http://localhost:64958/index.html?ui=P_0x1ecc00aa540_1&reconnect=auto" class="pyvis…

## 5. Set-cover selection

Pools every candidate this notebook produced per algorithm and runs `ransacov` over it.
At most `K` superquadrics are selected.

In [8]:
from point_cloud_utils import chamfer_distance
from scipy.spatial import cKDTree

from src.superquadrics import superquadric_sampling as samp


def evaluate(models, ref_points, all_points, n_clean, threshold):
    # Chamfer / Hausdorff / classification for a model set, mirroring pipeline.run_for_pc.
    meshes     = [supmesh.superquadric_mesh(m) for m in models]
    sampled, _ = samp.sampling_sq_random(meshes, n_points=4000, seed=pipeline.EVAL_SEED)
    est_pts    = np.vstack(sampled)

    d_ref_to_est = cKDTree(est_pts).query(ref_points, k=1)[0]
    d_est_to_ref = cKDTree(ref_points).query(est_pts, k=1)[0]

    mask = np.zeros(len(all_points), dtype=bool)
    for m in models:
        mask |= np.abs(pipeline.superquadric_radial_residual(m, all_points)) < threshold
    clf = (int(mask[:n_clean].sum()) + int((~mask[n_clean:]).sum())) / len(all_points)

    return {
        "chamfer": chamfer_distance(ref_points, est_pts),
        "cd_coverage": float(d_ref_to_est.mean()),
        "cd_accuracy": float(d_est_to_ref.mean()),
        "hausdorff": max(d_ref_to_est.max(), d_est_to_ref.max()),
        "classification_rate": clf,
    }


cover_threshold = pc_cfg.get("threshold", pipeline.THRESHOLD)
cover_rows, selected_by = [], {}

for algo in ALGORITHMS:
    candidates = [m for t in range(N_TRIALS) for m in models_by.get((t, algo), [])]
    if not candidates:
        continue

    selected_idx, n_covered = pipeline.ransacov(
        candidates, points, k=K, threshold=cover_threshold,
        residual_fn=lambda m, pts: np.abs(pipeline.superquadric_radial_residual(m, pts)),
    )
    if not selected_idx:
        print(f"  {algo:12s} -> nothing selected")
        continue

    chosen = [candidates[i] for i in selected_idx]
    selected_by[algo] = chosen
    metrics = evaluate(chosen, points_clean, points, n_clean, cover_threshold)
    cover_rows.append({"algo": algo, "k": len(chosen), "n_covered": n_covered,
                       "n_candidates": len(candidates), **metrics})
    print(f"  {algo:12s} k={len(chosen)}  covered={n_covered}/{len(points)}"
          f"  CD={metrics['chamfer']:.4f}  CLF={metrics['classification_rate']:.3f}")

cover_df = pd.DataFrame(cover_rows).round(6)
cover_df

  vanilla      k=5  covered=1396/1453  CD=0.1301  CLF=0.962
  lo-ransac    k=5  covered=1411/1453  CD=0.1142  CLF=0.971
  gc-ransac    k=5  covered=1331/1453  CD=0.0802  CLF=0.916
  gair-ransac  k=5  covered=1314/1453  CD=0.0426  CLF=0.911


,algo,k,n_covered,n_candidates,chamfer,cd_coverage,cd_accuracy,hausdorff,classification_rate
0,vanilla,5,1396,14,0.130147,0.014277,0.115870,0.725316,0.962147
1,lo-ransac,5,1411,13,0.114200,0.012452,0.101747,0.895846,0.971094
2,gc-ransac,5,1331,6,0.080201,0.013007,0.067194,0.555772,0.916036
3,gair-ransac,5,1314,14,0.042609,0.015784,0.026825,0.200726,0.911218


In [9]:
# visualise a set-cover selection (desktop window, as in section 4)
COVER_ALGO = "gair-ransac"

chosen = selected_by.get(COVER_ALGO)
if not chosen:
    print(f"no selection stored for {COVER_ALGO}")
else:
    meshes = [supmesh.superquadric_mesh(m) for m in chosen]
    colors = [PALETTE[i % len(PALETTE)] for i in range(len(meshes))]
    print(f"{COVER_ALGO} set-cover: {len(chosen)} superquadric(s)"
          " - close the window to continue")
    vis.show_mesh_and_points(meshes, pts=points_clean, point_size=5,
                             colors=colors, models=chosen)

gair-ransac set-cover: 5 superquadric(s) - close the window to continue


Widget(value='<iframe src="http://localhost:64958/index.html?ui=P_0x1ecc88eb560_2&reconnect=auto" class="pyvis…

Widget(value='<iframe src="http://localhost:64958/index.html?ui=P_0x1ecc0089220_3&reconnect=auto" class="pyvis…